In [1]:
# Test cell for data_client.py - run this in Jupyter to verify everything slaps

import pandas as pd
from datetime import datetime, timedelta
from data_client import data_client  # our unified client
import config  # loads .env toggle

print(f"Snoogans data provider locked: {config.DATA_PROVIDER.upper()}")
print(f"Mode: {config.MODE} | Balance start: ${config.STARTING_BALANCE}")

# Test 1: Pull recent SPY 1-min bars (last 100 mins or so)
end = datetime.now()
start = end - timedelta(days=2)  # plenty for intraday test

print("\nPulling SPY 1-min bars...")
spy_bars = data_client.get_spy_bars(start, end)
print(spy_bars.tail(10))  # show latest 10 rows
print(f"Got {len(spy_bars)} SPY bars — indicators ready")

# Test 2: Pull today's SPX option chain (for 0DTE test - note: greeks may be sparse on true 0DTE)
today_str = datetime.now().strftime("%Y-%m-%d")

print(f"\nPulling SPX 0DTE chain for {today_str}...")
chain_df = data_client.get_spx_option_chain(expiration_date=today_str)

if not chain_df.empty:
    # Clean up for display
    if 'greeks' in chain_df.columns:
        chain_df = pd.json_normalize(chain_df.to_dict(orient='records'))  # flatten if nested
    print(chain_df[['symbol', 'strike_price', 'option_type', 'bid_price', 'ask_price', 'delta', 'implied_volatility']].head(20))
    print(f"Pulled {len(chain_df)} contracts — delta hunt ready")
else:
    print("No 0DTE contracts today (weekend?) — try a future expiration for greeks test")

# Quick 30-delta finder test (puts example)
if not chain_df.empty and 'delta' in chain_df.columns:
    puts = chain_df[chain_df['option_type'] == 'put'].copy()
    if not puts.empty:
        puts['abs_delta'] = puts['delta'].abs()
        target = puts.iloc[(puts['abs_delta'] - 0.30).abs().argsort()[:1]]
        print(f"\nClosest 30-delta put: {target['symbol'].values[0]} | Delta: {target['delta'].values[0]:.3f} | Strike: {target['strike_price'].values[0]}")

print("\nData client test complete — Snoogans' eyes wide open.")
print("Next: wire the trend detector + strike selector for full internal paper runs Monday.")

Snoogans' eyes open — Polygon ready
